# Usable Stop Metrics

There are several experimental columns that should tell us more about how vehicle positions relate to stops for that trip. How complete are these columns in `fct_vehicle_locations`?
* Take `fct_vehicle_locations` -> aggregated to `trip_instance_key-stop_id-current_stop_sequence`
* We'd want to use potentially both `stop_id` and `current_stop_sequence` to understand whether the vp is at / in transit to a stop to better figure out when the bus has arrived at a stop, when is it dwelling, etc.

**Findings**
* Overall, not much is useful by stop, so let's retain all these experimental columns as arrays for `fct_vehicle_positions_trip_metrics`. Keep all the raw values.
* Also, add counts / distinct values for these experiemental columns. Be able to quickly take a pulse, see if the experimental columns are increasingly getting populated (through our outreach or operator provides it).

In [1]:
import gcsfs
import numpy as np
import pandas as pd

VP_GCS = "gs://calitp-analytics-data/data-analyses/rt_vehicle_positions/"

## Data Prep
Merge in scheduled trips and get a better overview by `service_date-route_id-direction_id`.

In [2]:
def clean_up_stop_metrics(df):
    """
    Catch stuff that should be part of the SQL
    """
    df = df.rename(
        columns = {"f0_": "congestion_level_array"}
    )
    return df

stop = pd.read_parquet(
    f"{VP_GCS}vp_stop_metrics.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    #filters = [[("gtfs_dataset_name", "==", "LA Metro Bus Vehicle Positions")]]
).pipe(clean_up_stop_metrics)

trips = pd.read_parquet(
    f"{VP_GCS}scheduled_trips.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    columns = ["trip_instance_key", "route_id", "direction_id"]
)

In [3]:
df = pd.merge(
    stop,
    trips,
    on = "trip_instance_key",
    how = "inner"
)

In [4]:
trip_cols = [
    "gtfs_dataset_name", "service_date", 
    "route_id", "direction_id", 
    "trip_instance_key", 
]

stop_cols = ["stop_id", "current_stop_sequence"]

### `current_status` and number of stops per trip captured 

Values: `in_transit_to`, `stopped_at`, `incoming_at`.

1. How many are null vs populated? Is this operator-specific or are nulls randomly distributed throughout?
2. Within those that are populated, does this match the number of stops that are being visited for that trip? `nunique(stop_id) <= nunique(current_stop_sequence)` or fairly close, because `stop_id` is allowed to repeat for a trip, especially if it loops back around.
3. If `current_status` is well populated, can we use `stopped_at` to find out when bus has arrived at a stop? If this tells us enough, we don't have to use trip updates to determine actual arrival. This would sit at a closer source (vp) to use for speedmaps.

**Exploratory Notes**

* Distribution of `current_status` is that nearly half is `IN_TRANSIT_TO`, 1/3 is `STOPPED_AT`. Very few are missing. This is promising.
   * Check whether every `stop_id` or `current_stop_sequence` shows up with `current_status`.
* Count unique stops (stop_id, stop_sequence) present in the vehicle_locations_df. Compare that how many stops show up with `current_status='STOPPED_AT'`.
   * Can we use `STOPPED_AT` to find when it arrived at a stop?
   * Aggregate this to trip (if trip serves 10 stops, how many of these 10 have `current_status=STOPPED_AT`)?
   * Aggregate to route-direction to see whether we have "complete" coverage of `current_status=STOPPED_AT` information within vehicle_locations_df. If trip is missing quite a bit, do we get enough coverage by route?

**Findings**
We're missing a lot, then this column isn't that usable now. 
* Significant variation in how good the coverage is.
* By route, we are missing quite a bit of stops, enough to not use it.
* By operators, some have min values that are negative. This just means we have to dedupe, because there are more rows with `STOPPED_AT` than we have unique stop_id / stop_sequence.

If we want to track how good that coverage is, we can add this to trip-level summary to get unique stop_id, stop_sequence, as well as non-null counts of `current_status`...and how many stop_ids or stop_sequence shows up with `current_status=STOPPED_AT`.

In [5]:
def stop_current_status_counts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Get counts of stop_id, current_stop_sequence, current_status for each trip.
    
    Visualize how trips are populated by operator-route_id-direction_id.
    """
    trip_cols = [
        "gtfs_dataset_name", "service_date", 
        "route_id", "direction_id", 
        "trip_instance_key", 
    ]
    
    current_status_pivoted_by_trip = df.pivot_table(
        index=trip_cols, 
        columns='current_status', 
        fill_value=0, 
        aggfunc='size'
    ).reset_index()
       
    counts_by_trip = (
        df
        .groupby(trip_cols, dropna=False)
        .agg({
            "stop_id": "nunique",
            "current_stop_sequence": "nunique",
        })
        .reset_index()
        .rename(columns = {
            "stop_id": "nunique_stop_id", 
            "current_stop_sequence": "nunique_stop_seq"
        })
    )

    df2 = pd.merge(
        counts_by_trip,
        current_status_pivoted_by_trip,
        on = trip_cols,
        how = "inner"
    )

    df2 = df2.assign(
        status_shortage = df2[["nunique_stop_id", "nunique_stop_seq"]].min(axis=1) - df2.STOPPED_AT
    )
    
    return df2


def usable_status_by_route(df: pd.DataFrame) -> pd.DataFrame:
    """
    """
    route_cols = ["service_date", "gtfs_dataset_name", "route_id"]
    counts_by_route = (
        df
        .groupby(route_cols, dropna=False)
        .agg({
            "trip_instance_key": "nunique",
            "status_shortage": "mean"}
        ).reset_index()
    )

    total_shortage_by_route = (
        df
        .groupby(route_cols, dropna=False)
        .agg({
            "status_shortage": "sum"}
        ).reset_index()
    )

    df2 = pd.merge(
        counts_by_route.rename(
            columns = {
                "status_shortage": "avg_stops_status_shortage", 
                "trip_instance_key": "n_trips"
            }),
        total_shortage_by_route.rename(columns = {"status_shortage": "total_stops_status_shortage"}),
        on = route_cols,
        how = "inner"
    )

    return df2

In [6]:
df.current_status.value_counts(dropna=False)

current_status
IN_TRANSIT_TO    1015041
STOPPED_AT        807274
INCOMING_AT       329325
None               45302
Name: count, dtype: int64

In [7]:
df.current_status.value_counts(dropna=False, normalize=True)

current_status
IN_TRANSIT_TO    0.462024
STOPPED_AT       0.367453
INCOMING_AT      0.149902
None             0.020620
Name: proportion, dtype: float64

In [8]:
cols_to_look_at = ["current_status"]
df[trip_cols + stop_cols + cols_to_look_at].sample(5)

,gtfs_dataset_name,service_date,route_id,direction_id,trip_instance_key,stop_id,current_stop_sequence,current_status
1103018,Bay Area 511 AC Transit Vehicle Position,2026-01-01,57,0,fefed2b09320447a963f73e49d3513dd,51375,23,IN_TRANSIT_TO
1065853,Bay Area 511 SamTrans VehiclePositions,2026-01-01,ECR,0,14f150a04c1ec0d4ba7d2fc7aed0e32e,336028,55,None
1748009,LA Metro Bus Vehicle Positions,2026-01-01,224-13196,1,476f91ff35650a2e677d68411dbea1dd,2838,53,IN_TRANSIT_TO
2123394,LA Metro Bus Vehicle Positions,2026-01-01,70-13196,1,bf8c18c0d97e9bf7ccd613db99b131d8,9997,11,IN_TRANSIT_TO
718666,Bay Area 511 AC Transit Vehicle Position,2026-01-01,840,1,f21993e33da6d1ae95cc32aab5c0b918,51548,6,INCOMING_AT


In [9]:
usable_status = stop_current_status_counts(df)
route_status = usable_status_by_route(usable_status)

In [10]:
route_status.sample(10)

,service_date,gtfs_dataset_name,route_id,n_trips,avg_stops_status_shortage,total_stops_status_shortage
161,2026-01-01,Bay Area 511 SamTrans VehiclePositions,ECR,126,70.388889,8869
202,2026-01-01,Bay Area 511 Tri Delta VehiclePositions,373,32,-6.625000,-212
205,2026-01-01,Bay Area 511 Tri Delta VehiclePositions,376,30,-3.866667,-116
234,2026-01-01,Foothill Vehicle Positions,10492,51,21.803922,1112
420,2026-01-01,Redding Vehicle Positions,24,4,1.000000,4
293,2026-01-01,LA Metro Bus Vehicle Positions,204-13196,172,11.220930,1930
18,2026-01-01,Bay Area 511 AC Transit Vehicle Position,34,34,42.735294,1453
166,2026-01-01,Bay Area 511 Santa Clara Transit VehiclePositions,203,2,11.000000,22
237,2026-01-01,Foothill Vehicle Positions,20187,64,18.593750,1190
453,2026-01-01,SCVTA Swiftly Vehicle Position,85,19,10.421053,198


In [11]:
route_status.groupby(["gtfs_dataset_name", "service_date"]).agg({
    "avg_stops_status_shortage": ["min", "mean", "max"],
    "total_stops_status_shortage": ["min", "mean", "max"]
}).reset_index().sample(10)

gtfs_dataset_name service_date  \
                                                                     
15                        Mendocino Vehicle Positions   2026-01-01   
13                    LA Metro Rail Vehicle Positions   2026-01-01   
12                     LA Metro Bus Vehicle Positions   2026-01-01   
3                 Bay Area 511 Marin VehiclePositions   2026-01-01   
5              Bay Area 511 SamTrans VehiclePositions   2026-01-01   
2   Bay Area 511 Golden Gate Transit Vehicle Posit...   2026-01-01   
0            Bay Area 511 AC Transit Vehicle Position   2026-01-01   
19                     SCVTA Swiftly Vehicle Position   2026-01-01   
6   Bay Area 511 Santa Clara Transit VehiclePositions   2026-01-01   
18                          Redding Vehicle Positions   2026-01-01   

   avg_stops_status_shortage                        \
                         min       mean        max   
15                  4.500000   4.500000   4.500000   
13                  1.019608   3.493036   9.703057   
12                  2.254717  16.405106  46.230769   
3                   3.564706   8.657776  14.291667   
5                   0.666667  27.600363  91.000000   
2                 -13.594595  -9.117735  -2.709677   
0                  12.481132  34.652611  69.538462   
19                  1.737374  16.524397  42.915789   
6                   1.818182  16.189640  41.600000   
18                  1.000000   1.000000   1.000000   

   total_stops_status_shortage                     
                           min         mean   max  
15                           9     9.000000     9  
13                         208   739.166667  2222  
12                          92  1388.165049  3980  
3                          127   247.142857   411  
5                            4  1323.384615  8869  
2                         -503  -296.750000   -84  
0                          239  2242.216667  7720  
19                           3  1019.500000  4861  
6                            3  1001.277778  4755  
18                           4     4.000000     4

### congestion_level
Values: `congestion_level_array`

1. Not clear from GTFS RT spec whether this is populated for every vehicle position or for the trip? Does it get updated when the congestion changes as the trip progresses?

   * It seems to be **by trip**, as within a trip, there is only 1 distinct value.
   * Although, maybe the default is set to `UNKNOWN_CONGESTION_LEVEL`, so there's not enough variation to tell. 

2. How many are null vs populated? Is this operator-specific or are nulls randomly distributed throughout?
   * Most of these are null or `UNKNOWN_CONGESTION_LEVEL`, so there's not much that's usable across all the operators here.
  
**Findings**
* It probably is set by trip, though we don't have enough variation to tell. 
* Most of these are null or `UNKNOWN_CONGESTION_LEVEL`, so there's not much that's usable across all the operators here.


In [12]:
trip_cols = [
    "gtfs_dataset_name", "service_date", 
    "route_id", "direction_id", 
    "trip_instance_key", 
]

stop_cols = ["stop_id", "current_stop_sequence"]
cols_to_look_at = ["congestion_level_array", "count_congestion_level"]

In [13]:
df[trip_cols + stop_cols + cols_to_look_at].count_congestion_level.value_counts()

count_congestion_level
0      2110388
1        59366
2        13335
3         4360
4         2393
        ...   
169          1
93           1
192          1
124          1
167          1
Name: count, Length: 162, dtype: Int64

In [14]:
df[trip_cols + stop_cols + cols_to_look_at].count_congestion_level.value_counts(normalize=True)

count_congestion_level
0      0.960603
1      0.027022
2       0.00607
3      0.001985
4      0.001089
         ...   
169         0.0
93          0.0
192         0.0
124         0.0
167         0.0
Name: proportion, Length: 162, dtype: Float64

In [15]:
# Figure out whether congestion_level_array is by vehicle or by trip or by stop
def aggregate_array_by_group(df: pd.DataFrame, group_cols: list, array_col: str) -> pd.DataFrame:
    """
    Groupby and combine the possible values in array.
    Get distinct values (np.unique), flatten the nested array into 1 single array 
    (np.concatenate for combining, ravel for flattening),
    and turn result into a list (.tolist()) instead of array.
    
    Replicate what Big Query ARRAY_AGG would do, but we aren't sorting here.
    """
    df2 = (
        df
        .groupby(group_cols, dropna=False)
        .agg({
            array_col: lambda x: np.unique(np.concatenate(np.asarray(x)).ravel()).tolist()
        })
        .reset_index()
    )

    # See how many distinct values were filled in
    # If unknown is filled in, tag a dummy
    df2 = df2.assign(
        distinct_array_values = df2.apply(lambda x: len(x[array_col]), axis=1),
        is_unknown = df2.apply(lambda x: "UNKNOWN_CONGESTION_LEVEL" in x[array_col], axis=1),
        is_zero = df2.apply(lambda x: 0 in x[array_col], axis=1),
    )

    return df2

In [16]:
# For a trip-stop, check whether there are different congestion_levels
# No, at most there is just 1 distinct value. It's all nulls or unknown.
congestion_by_stop = aggregate_array_by_group(df, trip_cols + stop_cols, "congestion_level_array")
congestion_by_stop.distinct_array_values.value_counts()

distinct_array_values
0    1360067
1      60939
Name: count, dtype: int64

In [17]:
congestion_by_stop[congestion_by_stop.distinct_array_values == 1].is_unknown.value_counts()

is_unknown
True    60939
Name: count, dtype: int64

In [18]:
# Repeat this for trip, if there are still just 1 distinct value per trip, then congestion_level is likely set per trip.
congestion_by_trip = aggregate_array_by_group(df, trip_cols, "congestion_level_array")
congestion_by_trip.distinct_array_values.value_counts()

distinct_array_values
0    38708
1     1459
Name: count, dtype: int64

In [19]:
congestion_by_trip[congestion_by_trip.distinct_array_values == 1].is_unknown.value_counts()

is_unknown
True    1459
Name: count, dtype: int64

### occupancy

Values: `occupancy_status`, `occupancy_percentage`

1. Not clear from the GTFS RT spec whether this is populated for every stop or vehicle position?
2. How many are null vs populated? Is this operator-specific or are nulls randomly distributed throughout?
3. If this is populated for stop or every vehicle position, what would be a meaningful metric? Occupancy at the stop to understand which stops are popular? Occupancy for the trip to understand ridership?

**Findings**
* still not clear whether this is populated for every stop, position, or trip. But, looking per trip, majority (90%+) is zero distinct values, which means it's
* by stop, about 85% of the values will have 0, 1, or 2 values filled in. so this is definitely not useful by stop or position. stop is granular of a row and `occupancy_status` isn't showing variation at that granularity (or finer granularity).
* by trip, Fresno, Long Beach provide some information here, but when you look at the stop information, it's null
* maybe add count of non-null and distinct values for trip, and if these columns are increasingly populated, we'll have a way to flag it. 

In [20]:
cols_to_look_at = ["count_occupancy_status", "occupancy_percentage_array"]

In [21]:
df[trip_cols + stop_cols + cols_to_look_at].count_occupancy_status.value_counts(dropna=False)

count_occupancy_status
0       996668
1       569875
2       293865
3       151069
4        59016
         ...  
316          1
417          1
651          1
1098         1
466          1
Name: count, Length: 413, dtype: Int64

In [22]:
# about 85% of the values will have 0, 1, or 2 values filled in
df[trip_cols + stop_cols + cols_to_look_at].count_occupancy_status.value_counts(dropna=False, normalize=True)

count_occupancy_status
0       0.453661
1       0.259395
2       0.133761
3       0.068763
4       0.026863
          ...   
316          0.0
417          0.0
651          0.0
1098         0.0
466          0.0
Name: proportion, Length: 413, dtype: Float64

In [23]:
# Take a look at the distinct values
occupancy_by_stop = aggregate_array_by_group(df, trip_cols + stop_cols, "occupancy_percentage_array")
occupancy_by_stop.distinct_array_values.value_counts()

distinct_array_values
0    1299951
1     120921
2        121
3         13
Name: count, dtype: int64

In [24]:
occupancy_by_stop.distinct_array_values.value_counts(normalize=True)

distinct_array_values
0    0.914810
1    0.085095
2    0.000085
3    0.000009
Name: proportion, dtype: float64

In [25]:
occupancy_by_stop[occupancy_by_stop.distinct_array_values > 0].is_zero.value_counts()

is_zero
True     120253
False       802
Name: count, dtype: int64

In [26]:
# Fresno, Long Beach provide some information here
occupancy_by_stop[(occupancy_by_stop.distinct_array_values > 1) & 
    (occupancy_by_stop.is_zero==False)].gtfs_dataset_name.value_counts()

gtfs_dataset_name
Fresno Vehicle Positions       73
Long Beach VehiclePositions    61
Name: count, dtype: int64

In [27]:
# Since it's already differing by stop, does it change from stop to stop?
# stop_id and current_stop_seq are nulls for where this is happening
occupancy_by_stop[
    (occupancy_by_stop.gtfs_dataset_name.isin(["Fresno Vehicle Positions", "Long Beach VehiclePositions"])) &
    (occupancy_by_stop.distinct_array_values > 1) &
    (occupancy_by_stop.is_zero==False)
][["gtfs_dataset_name", "trip_instance_key", 
   "route_id", "direction_id", 
   "stop_id", "current_stop_sequence"
  ]].drop_duplicates().dropna(subset=["stop_id", "current_stop_sequence"]).sort_values(["trip_instance_key", "current_stop_sequence"])

,gtfs_dataset_name,trip_instance_key,route_id,direction_id,stop_id,current_stop_sequence


In [28]:
# Take a look at the distinct values by trip, maybe it's differing here?
occupancy_by_trip = aggregate_array_by_group(df, trip_cols, "occupancy_percentage_array")
occupancy_by_trip.distinct_array_values.value_counts()

distinct_array_values
0    37442
1     2591
2      121
3       13
Name: count, dtype: int64

In [29]:
occupancy_by_trip[occupancy_by_trip.distinct_array_values > 0].is_zero.value_counts()

is_zero
True     1923
False     802
Name: count, dtype: int64

In [30]:
# At best, for Long Beach and Fresno, can flag which seem to be busy route-directions
occupancy_by_trip[
    (occupancy_by_trip.gtfs_dataset_name.isin(["Fresno Vehicle Positions", "Long Beach VehiclePositions"])) &
    (occupancy_by_trip.distinct_array_values > 1) &
    (occupancy_by_trip.is_zero==False)
][["gtfs_dataset_name", "trip_instance_key", 
   "route_id", "direction_id", "occupancy_percentage_array"
  ]].sort_values(["route_id", "direction_id"])

,gtfs_dataset_name,trip_instance_key,route_id,direction_id,occupancy_percentage_array
30184,Long Beach VehiclePositions,25e1d510b95ab5fddd45546b91701f44,111,0,"[100, 120]"
30192,Long Beach VehiclePositions,c3b492eb6f7a0548a5579b04405f7a39,111,0,"[100, 120]"
30198,Long Beach VehiclePositions,1061aea0c07107fc014beb06f10aae31,111,1,"[100, 120]"
30205,Long Beach VehiclePositions,91d169b8e9fcb9a97df20555ffaeeac6,111,1,"[100, 120]"
30225,Long Beach VehiclePositions,f68f97053f8855815c41fd039190a215,112,0,"[100, 120]"
...,...,...,...,...,...
31005,Long Beach VehiclePositions,e126fe422b0d1993702ac7eeeb98b902,61,1,"[100, 120]"
31026,Long Beach VehiclePositions,0d5000c9ca7d1f79d15de52f20e1e80b,71,1,"[20, 40]"
31081,Long Beach VehiclePositions,7e666104663ec1f274522fcdb92da971,94,0,"[20, 40]"
31087,Long Beach VehiclePositions,febe878e7c4b58adac06cf5d76b22806,94,0,"[20, 40]"


### header_message / vehicle_message age	
* Take a look at how Farhad has set this up for data quality checks
* These columns need to be carried from `fct_vehicle_locations` and arrays brought in for trip-level metrics.
* Right now, daily percentiles (25, 50 75, 90, 95, 99th percentiles) are calculated. But we cannot calculate percentiles without full arrays.

In [31]:
cols_to_look_at = ["avg_header_message_age", "avg_vehicle_message_age"]

In [32]:
df[df.trip_instance_key=="f8fa2a2674386c58bdcfff5f5ec51874"][trip_cols + stop_cols + cols_to_look_at].head()

,gtfs_dataset_name,service_date,route_id,direction_id,trip_instance_key,stop_id,current_stop_sequence,avg_header_message_age,avg_vehicle_message_age
0,Bay Area 511 Tri-Valley Wheels VehiclePositions,2026-01-01,30R,0,f8fa2a2674386c58bdcfff5f5ec51874,None,21,-2.750000,7.250000
131,Bay Area 511 Tri-Valley Wheels VehiclePositions,2026-01-01,30R,0,f8fa2a2674386c58bdcfff5f5ec51874,None,1,-2.090909,7.454545
220255,Bay Area 511 Tri-Valley Wheels VehiclePositions,2026-01-01,30R,0,f8fa2a2674386c58bdcfff5f5ec51874,None,12,-2.000000,3.000000
384820,Bay Area 511 Tri-Valley Wheels VehiclePositions,2026-01-01,30R,0,f8fa2a2674386c58bdcfff5f5ec51874,None,13,-2.200000,9.800000
384840,Bay Area 511 Tri-Valley Wheels VehiclePositions,2026-01-01,30R,0,f8fa2a2674386c58bdcfff5f5ec51874,None,23,-1.750000,6.500000


###  speed, odometer, bearing
Values: `speed: min, max, avg`

* How many are null vs populated? Is this operator-specific or are nulls randomly distributed throughout?
* Within those that are populated, what does distribution look like? Is it usable or is it capturing acceleration with blips? Can we average it and smooth it out?


Values: `odometer: min, max`

* How many are null vs populated? Is this operator-specific or are nulls randomly distributed throughout?
* Does this monotonically increase through trip progression? Is it accurately capturing the distance traveled between stops?
* Sanity check: how would a derived scheduled stop times + stop position -> stop's distance along shape -> difference between that would be odometer reading

Values: `position_bearing_array`

* How many are null vs populated? Is this operator-specific or are nulls randomly distributed throughout?
* What do values here look like?
* How would position bearing be translated to direction to be used in a meaningful way?

**Findings**
* Just bring in arrays for now, for trip, ordered by vehicle position for all these columns.
* Odometer seems to be null throughout, min/max didn't capture anything. Not useful for us, and without that, potentially could see if `speed` column could back out `odometer` based on timestamp deltas.
* Bearing potentially could be useful. `speed`, `bearing` would potentially be useful, though less so if we have `movingpandas` columns.
* There are nulls, so ARRAY_AGG should ignore those. Also get counts for non-null values for all these columns

In [33]:
trip_cols = [
    "gtfs_dataset_name", "service_date", 
    "route_id", "direction_id", 
    "trip_instance_key", 
]

stop_cols = ["stop_id", "current_stop_sequence"]
cols_to_look_at = ["min_speed", "avg_speed", "max_speed"]

In [34]:
df.max_speed.value_counts(dropna=False, normalize=True)

max_speed
0.000000     7.561055e-02
9.834880     1.421567e-02
8.940800     1.356613e-02
0.044704     1.265714e-02
8.046720     1.210774e-02
                 ...     
20.501978    4.551782e-07
1.105532     4.551782e-07
21.298330    4.551782e-07
1.091127     4.551782e-07
1.144629     4.551782e-07
Name: proportion, Length: 44014, dtype: float64

In [35]:
df.min_speed.value_counts(dropna=False, normalize=True)

min_speed
0.000000     2.333589e-01
0.044704     3.349155e-02
9.834880     9.850055e-03
8.940800     9.436298e-03
6.705600     8.842291e-03
                 ...     
1.340000     4.551782e-07
13.120000    4.551782e-07
13.179552    4.551782e-07
6.870000     4.551782e-07
22.995468    4.551782e-07
Name: proportion, Length: 40251, dtype: float64

In [36]:
df.avg_speed.value_counts(dropna=False, normalize=True)

avg_speed
0.000000    7.561055e-02
9.834880    8.819077e-03
0.044704    8.433541e-03
8.940800    8.255111e-03
NaN         7.674759e-03
                ...     
7.775246    4.551782e-07
8.444190    4.551782e-07
9.455568    4.551782e-07
0.837337    4.551782e-07
9.434830    4.551782e-07
Name: proportion, Length: 329476, dtype: float64

In [37]:
trip_cols = [
    "gtfs_dataset_name", "service_date", 
    "route_id", "direction_id", 
    "trip_instance_key", 
]

stop_cols = ["stop_id", "current_stop_sequence"]
cols_to_look_at = ["min_odometer", "max_odometer"]

In [38]:
df[trip_cols + stop_cols + cols_to_look_at].min_odometer.value_counts(dropna=False, normalize=True)

min_odometer
NaN    1.0
Name: proportion, dtype: float64

In [39]:
df[trip_cols + stop_cols + cols_to_look_at].max_odometer.value_counts(dropna=False, normalize=True)

max_odometer
NaN    1.0
Name: proportion, dtype: float64

In [40]:
bearing_by_trip = aggregate_array_by_group(df, trip_cols + stop_cols, "position_bearing_array")
bearing_by_trip.distinct_array_values.value_counts(normalize=True)

distinct_array_values
1      4.236583e-01
2      2.958805e-01
3      1.204632e-01
0      6.455286e-02
4      4.459235e-02
           ...     
169    7.037268e-07
198    7.037268e-07
133    7.037268e-07
134    7.037268e-07
146    7.037268e-07
Name: proportion, Length: 168, dtype: float64

In [41]:
bearing_by_trip.position_bearing_array.sample(10)

1287101                                       [358.4, 358.6]
888559     [0.0, 0.8, 3.0, 104.0, 178.6, 180.3, 182.0, 18...
103194                          [294.0, 297.0, 301.0, 306.0]
951263                                               [298.0]
429557                                               [255.0]
27810                    [210.0, 211.0, 212.0, 246.0, 249.0]
197000                                               [225.0]
94318                                  [138.0, 146.0, 147.0]
1378734                             [229.51, 229.73, 230.66]
72448                                         [141.0, 157.0]
Name: position_bearing_array, dtype: object